# W07 · Application 1 wrap-up: full Table 1 / 應用一收尾:完整 Table 1

**English.** Aggregate all methods across several Kodak images with all metrics
(PSNR, SSIM, LSD). We emphasize **LSD** (log-spectral distance): it measures
high-frequency fidelity, exactly where PEPS wins and plain PSNR under-reports.

**繁體中文.** 匯總多張 Kodak 上所有方法與所有指標(PSNR、SSIM、LSD)。特別強調
**LSD**(對數頻譜距離):量測高頻保真度,正是 PEPS 勝出、而純 PSNR 低估之處。

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import torch, matplotlib.pyplot as plt
from peps.train import auto_device
device = auto_device(); print('device', device)

device cuda


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [2]:
from apps.image.data import load_image, image_to_coords_targets, find_kodak
from apps.image.build import build_grid, build_grid_peps
from peps.train import fit, TrainConfig, render_full
from peps.metrics import psnr, ssim, lsd
import numpy as np
methods = {
  'grid':       lambda: build_grid(resolution=128, feature_dim=8),
  'grid_peps':  lambda: build_grid_peps(128, 8, 6, 'concat'),
  'pink_peps':  lambda: build_grid_peps(128, 8, 6, 'pink'),
}
images = [1, 5, 19]   # a few Kodak images (subset for speed)

## 1. Sweep methods x images / 掃方法 x 影像

In [3]:
results = {m: {'psnr': [], 'ssim': [], 'lsd': []} for m in methods}
for idx in images:
    img = load_image(find_kodak(idx), max_size=384)
    coords, targets, (H, W) = image_to_coords_targets(img)
    for m, builder in methods.items():
        model, pc = builder()
        fit(model, coords, targets, TrainConfig(steps=2000, batch_size=32768, lr=1e-2, device=device))
        pred = render_full(model, coords, device=device).reshape(H, W, 3).clamp(0, 1)
        results[m]['psnr'].append(psnr(pred, img))
        results[m]['ssim'].append(ssim(pred, img))
        results[m]['lsd'].append(lsd(pred, img))
    print('done image', idx)

done image 1


done image 5


done image 19


## 2. Table 1 summary / Table 1 匯總

In [4]:
print(f"{'method':12s} {'PSNR':>7s} {'SSIM':>7s} {'LSD':>7s}")
for m in methods:
    ps = np.mean(results[m]['psnr']); ss = np.mean(results[m]['ssim']); ls = np.mean(results[m]['lsd'])
    print(f'{m:12s} {ps:7.2f} {ss:7.4f} {ls:7.3f}')
print('\nLower LSD = better high-frequency fidelity (PEPS advantage).')

method          PSNR    SSIM     LSD
grid           37.00  0.9969   0.822
grid_peps      40.00  0.9981   0.573
pink_peps      41.09  0.9987   0.510

Lower LSD = better high-frequency fidelity (PEPS advantage).


## 3. Takeaway / 小結
PEPS variants lead on PSNR and especially LSD; Pink does it with fewer params.
This closes Application 1. Next: neural texture compression (W08).

PEPS 變體在 PSNR、尤其 LSD 領先;Pink 用更少參數達成。應用一到此結束,
接著:神經材質壓縮(W08)。